# 02 - Improved Two-Layer Deep CNN on Fashion-MNIST

An enhanced deep convolutional architecture featuring hierarchical feature extraction with dual convolution-pooling blocks and an intermediate dense projection layer.

### Key Architectural Upgrades over Baseline:
- **Hierarchical Convolutions**:
  - Block 1: `Conv2d(1 -> 32, 3x3, pad=1)` + `ReLU` + `MaxPool2d(2x2, s=2)` -> Output: `(32, 14, 14)`
  - Block 2: `Conv2d(32 -> 64, 3x3, pad=1)` + `ReLU` + `MaxPool2d(2x2, s=2)` -> Output: `(64, 7, 7)`
- **Expanded Dense Classifier**:
  - Dense Hidden Layer: `Linear(3136 -> 128)` + `ReLU`
  - Output Classifier: `Linear(128 -> 10)`
- **Trainable Parameters**: 421,642 (~13.4x more capacity than baseline)
- **Target Accuracy**: ~91.5% - 92.5% (+3% boost over baseline)
- **Model Checkpointing**: Serializes weights to `deep_fashion_cnn.pth`


In [ ]:
import sys
from pathlib import Path

# Add project root to path for src imports
root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

from src.dataset import get_data_loaders, FASHION_CLASSES
from src.models import DeepFashionCNN

In [ ]:
# 1. Dataset & DataLoader Setup
train_loader, test_loader, train_dataset, test_dataset = get_data_loaders(
    data_dir="../data" if Path.cwd().name == "notebooks" else "./data",
    batch_size=64
)

print(f"Pipeline active. Total training images loaded: {len(train_dataset):,}")
print(f"Total test images loaded: {len(test_dataset):,}")

In [ ]:
# 2. Two-Layer Deep CNN Architecture
deep_model = DeepFashionCNN()
print(deep_model)
total_params = sum(p.numel() for p in deep_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")

In [ ]:
# 3. Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(deep_model.parameters(), lr=0.001)

epochs = 5

for epoch in range(epochs):
    deep_model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        outputs = deep_model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (batch_idx + 1) % 300 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

In [ ]:
# 4. Evaluation on 10,000 Unseen Test Samples
deep_model.eval()
correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = deep_model(images)
        _, predicted = torch.max(outputs.data, 1)

        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

accuracy = 100 * correct_predictions / total_predictions
print(f'Deep CNN Accuracy on 10,000 unseen test images: {accuracy:.2f}%')

In [ ]:
# 5. Model Checkpoint Serialization
file_path = 'deep_fashion_cnn.pth'
torch.save(deep_model.state_dict(), file_path)
print(f"Success: {total_params:,} parameters saved to {file_path}")

In [ ]:
# 6. Visual Predictions Grid
dataiter = iter(test_loader)
images, labels = next(dataiter)

with torch.no_grad():
    outputs = deep_model(images)
    _, predicted = torch.max(outputs, 1)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.ravel()

for i in range(16):
    img = images[i].numpy()
    img = np.transpose(img, (1, 2, 0))

    true_label = FASHION_CLASSES[labels[i].item()]
    guess_label = FASHION_CLASSES[predicted[i].item()]

    text_color = 'green' if true_label == guess_label else 'red'

    axes[i].imshow(img.squeeze(), cmap='gray')
    axes[i].set_title(f"True: {true_label}\nGuess: {guess_label}", color=text_color, fontsize=10)
    axes[i].axis('off')

plt.subplots_adjust(hspace=0.5)
plt.show()